# Week 7 v2 — Notebook 1: build the three page sets

**What this notebook makes:** `w7_sets/` — for every page, five versions (`ai_edit`, `generation`, `copy_move`,
`splicing`, `untouched`) in two formats (PNG, JPEG q90), one box mask per page, a manifest, and quality checks.
Output is zipped and uploaded as the Kaggle dataset `PATH_TO_W7_COMMON_SETS_DATASET`.

**Sections 1–8 (N1-a)** build the page table, make the *untouched* pages match the AI edits exactly outside the box,
and check one box per page. **Sections 9–17 (N1-b)** add generation, copy-move, splicing, JPEG, edit quality, the
manifest, the checks, the QC figure and three zips.

> Run order: `SMOKE=True` → read the checks → `SMOKE=False` → Save & Run All. Accelerator: GPU T4 x2, Internet ON.

## 1. CONFIG

**Why (ELI5):** every setting lives in one place, so you never hunt through the code. `SMOKE=True` runs only 3 pages
per set, like tasting the soup before serving it. A fixed seed (42) means "random" choices come out the same each run.

In [ ]:
# standard tools for files, text, numbers and random choices
import os, re, io, sys, json, glob, shutil, random, time, zipfile, warnings
# tables, arrays and images
import numpy as np
import pandas as pd
from PIL import Image
# a tool that counts separate white blobs in a mask
from scipy import ndimage

# ======================= CONFIG: change settings here and nowhere else =======================
# True = run only a few pages to test everything; MUST be False for the real run
SMOKE = False
# how many pages per set in a smoke run (at least 3, because we make 3 size groups)
SMOKE_N = 3
# the one random seed used everywhere
SEED = 42
# where are we running? Kaggle has /kaggle/input; Colab has the google.colab module; otherwise it is a laptop
IN_KAGGLE = os.path.exists("/kaggle/input")
# Colab check
IN_COLAB = "google.colab" in sys.modules
# Kaggle: datasets in /kaggle/input, outputs in /kaggle/working
if IN_KAGGLE:
    # where datasets are mounted (both mount styles are searched below)
    SEARCH_ROOTS = ["/kaggle/input"]
    # where outputs go
    WORKING = "/kaggle/working"
# Colab: put the two input folders under /content (or Drive) named like the Kaggle slugs
elif IN_COLAB:
    # where to look
    SEARCH_ROOTS = ["/content", "/content/drive/MyDrive"]
    # where outputs go
    WORKING = "/content/w7_working"
# laptop: look in the current folder
else:
    # where to look
    SEARCH_ROOTS = [os.getcwd()]
    # where outputs go
    WORKING = os.path.join(os.getcwd(), "w7_working")
# optional: type the exact input folders here if automatic search fails (leave "" normally)
TC_DIR_OVERRIDE = ""
# the raw AnyText/DiffSTE folder
RAW_DIR_OVERRIDE = ""
# the folder that becomes the zip / dataset
OUT_DIR = os.path.join(WORKING, "w7_sets")
# scratch folder that is NOT zipped (rendered PDFs, page table)
WORK_DIR = os.path.join(WORKING, "w7_work")
# how many pages each set must have in a full run
EXPECTED_FULL = {"textctrl": 100, "anytext": 49, "diffste": 49}
# JPEG quality: saved once before the edit and once after
JPEG_QUALITY = 90
# generation: grow the old text pixels by this many pixels before erasing (same as Week 7 notebook 06)
OTSU_DILATE = 1
# generation: inpaint radius in pixels (same as Week 7 notebook 06)
INPAINT_RADIUS = 3
# donor words: EasyOCR must be at least this sure (same 70% rule as Week 4)
OCR_MIN_CONF = 0.7
# donors: largest allowed RGB distance between the ink colours (max possible is about 441)
COLOUR_TOL = 60
# donors: text height tolerance tiers (first 10%, then 20%)
HEIGHT_TIERS = [("10%", 0.10), ("20%", 0.20)]
# the five versions of every page
VERSIONS = ["ai_edit", "generation", "copy_move", "splicing", "untouched"]
# =============================================================================================

# pages per set that this run must contain
EXPECTED = {k: (SMOKE_N if SMOKE else v) for k, v in EXPECTED_FULL.items()}
# make python's random numbers repeatable
random.seed(SEED)
# make numpy's random numbers repeatable
np.random.seed(SEED)
# create the output and scratch folders
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
# save the settings next to the results so we can always see how the run was made
config = dict(SMOKE=SMOKE, SMOKE_N=SMOKE_N, SEED=SEED, JPEG_QUALITY=JPEG_QUALITY, EXPECTED=EXPECTED)
# write config.json
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

# the list that collects every check (name, value, passed, hard); written to checks.csv
CHECKS = []

# add one check to the list
def check(name, value, passed, hard=True):
    # store the check
    CHECKS.append(dict(check=name, value=value, passed=bool(passed), hard=hard))
    # show it right away
    print(("PASS " if passed else ("FAIL " if hard else "WARN ")) + name + " -> " + str(value))

# stop the notebook with a clear message if any hard check failed
def stop_if_failed():
    # collect the hard failures
    failed = [c["check"] for c in CHECKS if c["hard"] and not c["passed"]]
    # stop loudly
    if failed:
        raise RuntimeError("STOP: these checks failed: " + "; ".join(failed))

# a big warning while SMOKE is on
if SMOKE:
    print("#" * 78)
    print("#  SMOKE=True  ->  only", SMOKE_N, "pages per set.  Set SMOKE=False before Save & Run All.")
    print("#" * 78)
# show the settings
print(config)

## 2. Environment

**Why (ELI5):** some input pages are PDFs. A PDF is a recipe, not a picture; we need a tool (PyMuPDF) that cooks the
recipe into pixels. This cell makes sure it is installed.

In [ ]:
# try to load PyMuPDF (the PDF-to-picture tool)
try:
    # new import name
    import pymupdf
# if it is missing, install it and try again
except ImportError:
    # run pip from python
    import subprocess, sys
    # install quietly
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
    # import it now
    import pymupdf
# show the version so the run is documented
print("PyMuPDF version:", pymupdf.__doc__.split(":")[0] if pymupdf.__doc__ else "unknown")

## 3. Find the input datasets

**Why (ELI5):** Kaggle can put a dataset in two different places depending on how it was added. This cell looks in
both, up to 4 folders deep, for a marker we know is inside (`plan.csv` for TextCtrl, `doc_input_anytext_diffste` for
the AnyText/DiffSTE raw data).

In [ ]:
# look for a folder that holds a marker file/folder, in both Kaggle mount styles
def find_dir(slug, marker):
    # every place the dataset could be mounted
    bases = []
    # go through each search root
    for root in SEARCH_ROOTS:
        # style 1: /kaggle/input/<slug>
        bases.append(os.path.join(root, slug))
        # style 2: /kaggle/input/datasets/<user>/<slug>
        bases += glob.glob(os.path.join(root, "datasets", "*", slug))
    # try each candidate base folder
    for base in bases:
        # skip the ones that do not exist
        if not os.path.isdir(base):
            continue
        # walk down through the folders
        for folder, subfolders, files in os.walk(base):
            # how many levels below the base we are
            depth = folder[len(base):].count(os.sep)
            # found the marker here (as a file or as a subfolder)
            if marker in files or marker in subfolders:
                return folder
            # do not go deeper than 4 levels
            if depth >= 4:
                subfolders[:] = []
    # nothing found: show what exists (at most 20 names) and stop
    seen = []
    # list the top of each search root
    for root in SEARCH_ROOTS:
        # only if it exists
        if os.path.isdir(root):
            seen += os.listdir(root)[:20]
    # stop with a clear message
    raise FileNotFoundError("Could not find " + marker + " for dataset " + slug + ". Found in input: " + str(seen[:20]))

# the TextCtrl run folder (holds plan.csv, originals/, tampered/, masks/box/)
TC_DIR = TC_DIR_OVERRIDE or find_dir("PATH_TO_TEXTCTRL_RUN_DATASET", "plan.csv")
# the AnyText/DiffSTE raw folder (holds doc_input_..., gt_masks_..., outputs_...)
RAW_DIR = RAW_DIR_OVERRIDE or find_dir("anytext-diffste-raw", "doc_input_anytext_diffste")
# print the two folders
print("TextCtrl folder :", TC_DIR)
print("Raw folder      :", RAW_DIR)
# count the files in each sub-folder we use (counts only, no listing)
for sub in ["originals", "tampered", "masks/box"]:
    print("  textctrl", sub, len(os.listdir(os.path.join(TC_DIR, sub))))
# same for the raw folder
for sub in ["doc_input_anytext_diffste", "gt_masks_diffste_anytext", "outputs_anytext_50", "outputs_diffste_50"]:
    print("  raw", sub, len(os.listdir(os.path.join(RAW_DIR, sub))))

## 4. Helpers and the kind rule

**Why (ELI5):** copy-move and splicing must paste a *number* over a number and a *word* over a word. So every edited
text gets a label: `number` or `word`.

**Rule (from CLAUDE.md):** drop spaces and a leading `RM` or `IDR`. Call it a `number` if at least half of what is left
are digits and everything else is one of `/ . , % - : +`. Otherwise `word`.

Tiny example: `RM 1,250.00` → `1,250.00` → 6 of 8 characters are digits, the rest are `,` and `.` → **number**.
`SGD` → 0 digits → **word**.

In [ ]:
# the symbols allowed inside a number besides digits
NUMBER_SYMBOLS = set("/.,%-:+")

# label a piece of text as "number" or "word"
def text_kind(text):
    # remove all spaces
    t = str(text).replace(" ", "")
    # remove one leading currency prefix
    for prefix in ("RM", "IDR"):
        # only if the text starts with it (ignoring capitals)
        if t.upper().startswith(prefix):
            # cut it off
            t = t[len(prefix):]
            # one prefix is enough
            break
    # nothing left means it is not a number
    if len(t) == 0:
        return "word"
    # count the digits
    digits = sum(ch.isdigit() for ch in t)
    # every character must be a digit or an allowed symbol
    only_allowed = all(ch.isdigit() or ch in NUMBER_SYMBOLS for ch in t)
    # at least one digit, at least half digits, nothing else
    if digits >= 1 and digits * 2 >= len(t) and only_allowed:
        return "number"
    # everything else is a word
    return "word"

# list files in a folder by the first number in their name (handles Img2.pdf.png, im42.png, img1_edited.png)
def files_by_number(folder):
    # the answer: number -> file name
    found = {}
    # look at every visible file
    for name in sorted(os.listdir(folder)):
        # skip hidden files such as .DS_Store
        if name.startswith("."):
            continue
        # find the first run of digits in the name
        m = re.search(r"(\d+)", name)
        # no number means we cannot match it
        if m is None:
            continue
        # turn it into an integer
        k = int(m.group(1))
        # two files with the same number would be ambiguous: stop
        if k in found:
            raise RuntimeError("Two files share the number " + str(k) + " in " + folder + ": " + found[k] + ", " + name)
        # remember it
        found[k] = name
    # give back the table
    return found

# open an image as plain RGB (transparent parts become white)
def load_rgb(path):
    # open the file
    img = Image.open(path)
    # pictures with transparency: paste on white
    if img.mode in ("RGBA", "LA", "P"):
        # convert to RGBA first
        img = img.convert("RGBA")
        # a white page
        white = Image.new("RGBA", img.size, (255, 255, 255, 255))
        # put the picture on it
        img = Image.alpha_composite(white, img)
    # return plain RGB
    return img.convert("RGB")

# read a mask and describe its white box
def box_from_mask(path):
    # white = above 127
    white = np.array(Image.open(path).convert("L")) > 127
    # count separate white blobs
    _, blobs = ndimage.label(white)
    # where the white pixels are
    ys, xs = np.where(white)
    # an empty mask has no box
    if len(xs) == 0:
        return dict(x1=0, y1=0, x2=0, y2=0, blobs=0, is_rectangle=False, mask_w=white.shape[1], mask_h=white.shape[0])
    # the bounding rectangle (x2, y2 are exclusive)
    x1, x2, y1, y2 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1
    # a perfect rectangle has no holes: white count equals rectangle area
    is_rect = int(white.sum()) == (x2 - x1) * (y2 - y1)
    # return everything we learned
    return dict(x1=x1, y1=y1, x2=x2, y2=y2, blobs=int(blobs), is_rectangle=bool(is_rect), mask_w=white.shape[1], mask_h=white.shape[0])

# count pixels that differ between two pages OUTSIDE the box
def outside_diff(a, b, box):
    # largest colour difference per pixel
    d = np.abs(a.astype(np.int16) - b.astype(np.int16)).max(axis=2)
    # ignore the inside of the box
    d[box["y1"]:box["y2"], box["x1"]:box["x2"]] = 0
    # number of changed pixels, number changed by more than 8 levels, biggest change
    return int((d > 0).sum()), int((d > 8).sum()), int(d.max())

# count pixels that differ between two pages INSIDE the box (an AI edit must change something there)
def inside_diff(a, b, box):
    # largest colour difference per pixel
    d = np.abs(a.astype(np.int16) - b.astype(np.int16)).max(axis=2)
    # only the box
    return int((d[box["y1"]:box["y2"], box["x1"]:box["x2"]] > 0).sum())

# a few tiny examples so we can see the rule working
examples = ["3700055341", "RM 1,250.00", "12/05/2025", "IDR500", "SGD", "Limited", "Day", "-", "A1"]
# show text -> kind
print(pd.DataFrame({"text": examples, "kind": [text_kind(t) for t in examples]}).to_string(index=False))

## 5. The page table

**Why (ELI5):** one table that says, for every page: where its files are, where the edit box is, what text was typed,
which kind it is, and how big the box is. Everything later reads this table, so we build it once and check it.

- **Box:** TextCtrl = from `plan.csv`; AnyText/DiffSTE = bounding rectangle of the white pixels in `gt_masks`.
- **Box share:** box area ÷ page area. **Size groups:** three equal groups (small/medium/large) *within each set*.
- AnyText and DiffSTE share the same 49 pages, boxes and new text (set `shared49` for the non-AI versions).

In [ ]:
# ---------- TextCtrl ----------
# read plan.csv with every column as text (so "007" keeps its zeros)
plan = pd.read_csv(os.path.join(TC_DIR, "plan.csv"), dtype=str, keep_default_na=False)
# the rows we build
rows = []
# the TextCtrl pages that are incomplete
missing = {"textctrl": [], "shared49": []}
# go through every planned page
for r in plan.to_dict("records"):
    # the page id, e.g. page_0001
    page = r["page_id"]
    # the three files this page needs
    paths = dict(
        untouched_src=os.path.join(TC_DIR, "originals", page + ".png"),
        ai_src=os.path.join(TC_DIR, "tampered", page + ".png"),
        mask_src=os.path.join(TC_DIR, "masks", "box", page + ".png"),
    )
    # note which files are missing
    absent = [k for k, p in paths.items() if not os.path.exists(p)]
    # a page with a missing file cannot be used
    if absent:
        missing["textctrl"].append(page + ": " + ",".join(absent))
        continue
    # one table row for this page
    rows.append(dict(set="textctrl", page=page, editor="textctrl", num=int(page.split("_")[1]),
                     x1=int(r["x1"]), y1=int(r["y1"]), x2=int(r["x2"]), y2=int(r["y2"]),
                     new_text=r["target_text"], source_text=r["source_text"], plan_kind=r["kind"], **paths))

# ---------- AnyText and DiffSTE ----------
# the raw folders
doc_dir = os.path.join(RAW_DIR, "doc_input_anytext_diffste")
gt_dir = os.path.join(RAW_DIR, "gt_masks_diffste_anytext")
any_dir = os.path.join(RAW_DIR, "outputs_anytext_50")
dif_dir = os.path.join(RAW_DIR, "outputs_diffste_50")
# match files by the number in their name
docs, gts, anys, difs = files_by_number(doc_dir), files_by_number(gt_dir), files_by_number(any_dir), files_by_number(dif_dir)
# the new text for every id (kept as text)
nt = pd.read_csv(os.path.join(RAW_DIR, "new_text.csv"), dtype=str, keep_default_na=False)
# id -> new text
new_text = {int(i): t for i, t in zip(nt["id"], nt["new_text"])}
# page numbers that appear anywhere
all_nums = sorted(set(new_text) | set(docs) | set(gts) | set(anys) | set(difs))
# keep only numbers that have every file
shared_nums = []
# check each number
for k in all_nums:
    # which of the five inputs are missing for this number
    absent = [n for n, d in [("new_text", new_text), ("input", docs), ("gt_mask", gts), ("anytext_out", anys), ("diffste_out", difs)] if k not in d]
    # incomplete: log it
    if absent:
        missing["shared49"].append("img%02d: %s" % (k, ",".join(absent)))
    # complete: keep it
    else:
        shared_nums.append(k)
# SMOKE: use only the first SMOKE_N complete pages of the shared set
if SMOKE:
    shared_nums = shared_nums[:SMOKE_N]
# SMOKE: use only the first SMOKE_N TextCtrl pages
if SMOKE:
    rows = rows[:SMOKE_N]

# the box of each shared page comes from its gt mask
for k in shared_nums:
    # page id such as img07
    page = "img%02d" % k
    # the mask file
    mask_path = os.path.join(gt_dir, gts[k])
    # its white rectangle
    b = box_from_mask(mask_path)
    # one row for each editor (same page, same box, same new text)
    for editor, d, folder in [("anytext", anys, any_dir), ("diffste", difs, dif_dir)]:
        # add the row
        rows.append(dict(set=editor, page=page, editor=editor, num=k, x1=b["x1"], y1=b["y1"], x2=b["x2"], y2=b["y2"],
                         new_text=new_text[k], source_text="", plan_kind="",
                         untouched_src=os.path.join(doc_dir, docs[k]), ai_src=os.path.join(folder, d[k]), mask_src=mask_path))

# make the table
pages = pd.DataFrame(rows)
# every AI output must be a readable picture (a PDF or a damaged file stops us here with a clear message)
unreadable = []
# try each AI output
for p in pages["ai_src"]:
    # open it
    try:
        # reading the size is enough to know it is a picture
        Image.open(p).size
    # not a picture
    except Exception:
        # remember the file name
        unreadable.append(os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)))
# stop with an explanation
if unreadable:
    # the message
    raise RuntimeError("STOP: " + str(len(unreadable)) + " AI output files are not readable pictures (for example PDFs): " + str(unreadable[:8])
                       + ". The folders outputs_anytext_50 / outputs_diffste_50 must hold the EDITED images (e.g. img7_edited.png), not copies of the input pages. Fix the dataset and run again.")
# page size = size of the AI output (this is what everything must match)
sizes = [Image.open(p).size for p in pages["ai_src"]]
# width and height columns
pages["W"] = [s[0] for s in sizes]
pages["H"] = [s[1] for s in sizes]
# the kind from our rule
pages["kind"] = [text_kind(t) for t in pages["new_text"]]
# share of the page covered by the box
pages["box_share"] = (pages["x2"] - pages["x1"]) * (pages["y2"] - pages["y1"]) / (pages["W"] * pages["H"])
# three equal groups within each set (rank with ties broken by order so groups stay equal)
pages["size_group"] = pages.groupby("set")["box_share"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 3, labels=["small", "medium", "large"]).astype(str))
# save the table (not part of the dataset)
pages.to_csv(os.path.join(WORK_DIR, "page_table.csv"), index=False)

# ---------- checks on counts ----------
# every set must have the expected number of pages
for s, n in EXPECTED.items():
    # count this set
    got = int((pages["set"] == s).sum())
    # hard check
    check("pages in set " + s, str(got) + " (expected " + str(n) + ")", got == n)
# incomplete input pages: hard in a full run, only a warning in SMOKE (SMOKE skips them)
n_missing = len(missing["textctrl"]) + len(missing["shared49"])
# show the first few
check("input pages with a missing file", str(n_missing) + " " + str((missing["textctrl"] + missing["shared49"])[:8]), n_missing == 0, hard=not SMOKE)
# AnyText and DiffSTE outputs must have the same size on every shared page
size_mismatch = []
# compare page by page
for k in shared_nums:
    # the two sizes
    sa, sd = Image.open(os.path.join(any_dir, anys[k])).size, Image.open(os.path.join(dif_dir, difs[k])).size
    # a difference means no single shared untouched page can exist
    if sa != sd:
        size_mismatch.append("img%02d %s vs %s" % (k, sa, sd))
# hard check
check("AnyText and DiffSTE outputs same size", str(len(size_mismatch)) + " mismatches " + str(size_mismatch[:5]), len(size_mismatch) == 0)
# the kind rule vs TextCtrl's own kind column (information only)
tc = pages[pages["set"] == "textctrl"]
# how many disagree
disagree = int((tc["kind"] != tc["plan_kind"]).sum())
# show it
check("kind rule vs plan.csv kind (disagreements)", str(disagree) + " of " + str(len(tc)), disagree == 0, hard=False)
# stop here if a hard check failed
stop_if_failed()

# ---------- summary table ----------
# per set: pages, kinds, size groups, box share
summary = pages.groupby("set").agg(pages=("page", "count"), numbers=("kind", lambda s: int((s == "number").sum())),
                                   words=("kind", lambda s: int((s == "word").sum())),
                                   box_share_min=("box_share", "min"), box_share_median=("box_share", "median"),
                                   box_share_max=("box_share", "max"))
# print it
print(summary.round(5).to_string())
# first rows of the table (no image paths, to keep it short)
print(pages[["set", "page", "x1", "y1", "x2", "y2", "W", "H", "new_text", "kind", "box_share", "size_group"]].head(8).to_string(index=False))

## 6. Render the PDF inputs at the AI output's exact size

**Why (ELI5):** to say "nothing outside the box changed" we must lay the original page and the AI page on top of each
other pixel by pixel. That only works if both are the same size. Some originals are PDFs, so we draw each PDF at
exactly the AI output's width and height (we scale to the right width, then nudge by a pixel or two if needed and
report the gap). PNG inputs of a different size are resized the same way.

In [ ]:
# folder for the rendered originals (cache, not zipped)
RENDER_DIR = os.path.join(WORK_DIR, "untouched_render")
# create it
os.makedirs(RENDER_DIR, exist_ok=True)

# draw the first page of a PDF at exactly (w, h) pixels; also return the height before nudging
def render_pdf(path, w, h):
    # open the PDF
    doc = pymupdf.open(path)
    # its first page
    page = doc[0]
    # the zoom that makes the width come out right
    zoom = w / page.rect.width
    # draw the page without transparency
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False)
    # turn the pixels into a PIL image
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    # remember the height we got before nudging
    h_before = pix.height
    # nudge to exactly the wanted size if needed
    if img.size != (w, h):
        img = img.resize((w, h), Image.LANCZOS)
    # close the PDF
    doc.close()
    # give back the image and the height gap
    return img, h_before

# one row of information per shared page
render_rows = []
# the shared pages: one row per page (use the anytext rows)
shared = pages[pages["set"] == "anytext"]
# go page by page
for r in shared.to_dict("records"):
    # where the rendered original is cached
    out_path = os.path.join(RENDER_DIR, r["page"] + ".png")
    # is the input a PDF?
    is_pdf = r["untouched_src"].lower().endswith(".pdf")
    # the wanted size
    w, h = int(r["W"]), int(r["H"])
    # PDF: render at the exact size
    if is_pdf:
        # only do the slow work if not cached
        if os.path.exists(out_path):
            # read the size of the cached file; the height gap is not known any more
            gap = np.nan
        else:
            # render
            img, h_before = render_pdf(r["untouched_src"], w, h)
            # save it as PNG
            img.save(out_path)
            # how many pixels of height we had to nudge
            gap = h_before - h
        # input size is "pdf"
        in_size = "pdf"
    # PNG: copy at exact size (resize only if the sizes differ)
    else:
        # only if not cached
        if not os.path.exists(out_path):
            # open it
            img = load_rgb(r["untouched_src"])
            # the size we found
            in_size = img.size
            # resize if different
            if img.size != (w, h):
                img = img.resize((w, h), Image.LANCZOS)
            # save
            img.save(out_path)
        # cached: read the size from the original file
        else:
            in_size = Image.open(r["untouched_src"]).size
        # a PNG has no nudge, we report the largest size difference (width or height) instead
        gap = max(abs(in_size[0] - w), abs(in_size[1] - h))
    # store what happened
    render_rows.append(dict(page=r["page"], input_type="pdf" if is_pdf else "png", input_size=str(in_size),
                            out_size=str((w, h)), size_gap_px=gap))

# make the info table
render_info = pd.DataFrame(render_rows)
# show a summary: how many pdf / png, how many needed nudging or resizing
print(render_info.groupby("input_type").size().to_string())
# pages whose size had to be changed by more than 2 pixels (should be none)
big_gap = render_info[render_info["size_gap_px"].abs() > 2]
# a big gap means the aspect ratio differs: show it
print("pages with a size gap above 2 px:", len(big_gap))
# show the table (it is short)
print(render_info.to_string(index=False))
# hard check: no large gaps
check("rendered originals within 2 px of the AI size", str(len(big_gap)) + " pages " + str(list(big_gap["page"])[:8]), len(big_gap) == 0)
# stop if a render is too far off
stop_if_failed()

## 7. Outside-the-box check and the rebuild

**Why (ELI5):** a detector must find the *edit*, not a leftover difference elsewhere on the page (different
renderer, different blur, the editor redrawing everything). So we count how many pixels outside the box differ between
the untouched page and each AI output.

**Rule:** if any pixel outside the box differs, the untouched page is **rebuilt** = the AI output with the original
pasted back inside the box only. Then nothing outside the box differs.

**AnyText and DiffSTE share ONE untouched page**, but each editor changes the page its own way. So: the rebuilt
untouched page is made from the **AnyText** output; the **DiffSTE** edit is then the rebuilt page with only
DiffSTE's *box pixels* pasted in. The box pixels are exactly what DiffSTE drew. TextCtrl needs no rebuild because its
tampered page is the original with the crop pasted in.

In [ ]:
# final image folders follow the contract: images/<set>/<version>/<format>/<page>.png
def image_path(set_name, version, page, fmt="png"):
    # build and create the folder
    folder = os.path.join(OUT_DIR, "images", set_name, version, fmt)
    # make sure it exists
    os.makedirs(folder, exist_ok=True)
    # file name
    return os.path.join(folder, page + "." + ("jpg" if fmt == "jpg" else "png"))

# statistics per page and editor (before any rebuild)
stat_rows = []
# which shared pages needed the rebuild
rebuilt_pages = {}
# which shared pages needed DiffSTE patched
patched_pages = {}
# AI edits that changed nothing inside the box (the file may be a copy of the input page)
no_edit = []

# ---------- TextCtrl: copy, no rebuild needed ----------
# go page by page
for r in pages[pages["set"] == "textctrl"].to_dict("records"):
    # open the original and the tampered page
    a, b = np.array(load_rgb(r["untouched_src"])), np.array(load_rgb(r["ai_src"]))
    # same size is required before comparing
    if a.shape != b.shape:
        # record a size problem; the final checks will fail on it
        stat_rows.append(dict(set="textctrl", page=r["page"], changed_px=-1, changed_gt8=-1, max_diff=-1))
        continue
    # pixels that changed outside the box
    n, n8, mx = outside_diff(a, b, r)
    # an AI edit that changed nothing inside the box is not an edit
    if inside_diff(a, b, r) == 0:
        # remember it
        no_edit.append("textctrl/" + r["page"])
    # store
    stat_rows.append(dict(set="textctrl", page=r["page"], changed_px=n, changed_gt8=n8, max_diff=mx))
    # write the two final PNGs (decoded pixels, lossless)
    Image.fromarray(a).save(image_path("textctrl", "untouched", r["page"]))
    # the AI edit
    Image.fromarray(b).save(image_path("textctrl", "ai_edit", r["page"]))
    # no rebuild for TextCtrl
    rebuilt_pages[r["page"]] = False

# ---------- AnyText / DiffSTE ----------
# the rows of each editor, keyed by page
any_rows = {r["page"]: r for r in pages[pages["set"] == "anytext"].to_dict("records")}
dif_rows = {r["page"]: r for r in pages[pages["set"] == "diffste"].to_dict("records")}
# go page by page
for page, ra in any_rows.items():
    # the DiffSTE row for the same page
    rd = dif_rows[page]
    # the rendered original, the AnyText output and the DiffSTE output
    o = np.array(Image.open(os.path.join(RENDER_DIR, page + ".png")).convert("RGB"))
    a = np.array(load_rgb(ra["ai_src"]))
    d = np.array(load_rgb(rd["ai_src"]))
    # how many pixels differ outside the box for each editor (before rebuilding)
    na, na8, ma = outside_diff(o, a, ra)
    nd, nd8, md_ = outside_diff(o, d, ra)
    # each editor must change something inside the box
    if inside_diff(o, a, ra) == 0:
        # AnyText did not change the box
        no_edit.append("anytext/" + page)
    # DiffSTE
    if inside_diff(o, d, ra) == 0:
        # remember
        no_edit.append("diffste/" + page)
    # store the statistics
    stat_rows.append(dict(set="anytext", page=page, changed_px=na, changed_gt8=na8, max_diff=ma))
    stat_rows.append(dict(set="diffste", page=page, changed_px=nd, changed_gt8=nd8, max_diff=md_))
    # does this page need a rebuild? (any difference outside the box for either editor)
    need = (na > 0) or (nd > 0)
    # the box as slices
    ys, xs = slice(ra["y1"], ra["y2"]), slice(ra["x1"], ra["x2"])
    # rebuilt untouched = AnyText output with the original pasted back inside the box
    if need:
        # copy the AnyText output
        base = a.copy()
        # paste the original box content back
        base[ys, xs] = o[ys, xs]
    # no rebuild: the untouched page is just the rendered original
    else:
        base = o
    # DiffSTE final = untouched with only DiffSTE's box pixels pasted in
    d_final = base.copy()
    # paste DiffSTE's box
    d_final[ys, xs] = d[ys, xs]
    # was DiffSTE's page changed by this (differs from its raw output outside the box)?
    patched = int((np.abs(d_final.astype(np.int16) - d.astype(np.int16)).max(axis=2) > 0).sum()) > 0
    # AnyText final = untouched with only AnyText's box pixels pasted in (equal to the raw output when rebuilt from it)
    a_final = base.copy()
    # paste AnyText's box
    a_final[ys, xs] = a[ys, xs]
    # remember what happened
    rebuilt_pages[page] = bool(need)
    # DiffSTE patched flag
    patched_pages[page] = bool(patched)
    # write the three PNGs
    Image.fromarray(base).save(image_path("shared49", "untouched", page))
    # AnyText edit
    Image.fromarray(a_final).save(image_path("anytext", "ai_edit", page))
    # DiffSTE edit
    Image.fromarray(d_final).save(image_path("diffste", "ai_edit", page))

# ---------- report ----------
# the statistics table
stats = pd.DataFrame(stat_rows)
# per set: pages with any change, with a change above 8 levels, the largest change, and the median share changed
rep = stats.groupby("set").agg(pages=("page", "count"), pages_any_change=("changed_px", lambda s: int((s > 0).sum())),
                               pages_change_gt8=("changed_gt8", lambda s: int((s > 0).sum())),
                               max_diff=("max_diff", "max"), median_changed_px=("changed_px", "median"))
# print it
print("BEFORE rebuilding (untouched vs raw AI output, outside the box):")
print(rep.to_string())
# how many shared pages were rebuilt, and how many DiffSTE pages were patched
n_rebuilt = sum(v for p, v in rebuilt_pages.items() if p.startswith("img"))
# patched count
n_patched = sum(patched_pages.values())
# every AI edit must differ from the untouched page inside the box
check("AI edits that changed nothing inside the box", str(len(no_edit)) + " " + str(no_edit[:8]), len(no_edit) == 0)
# stop now with an explanation if any file is a copy of the input
if no_edit:
    # the message
    raise RuntimeError("STOP: " + str(len(no_edit)) + " AI outputs are identical to the input inside the box (copies of the input page, not edits): " + str(no_edit[:8])
                       + ". Fix outputs_anytext_50 / outputs_diffste_50 in the dataset and run again.")
# print them
print("shared pages with a rebuilt untouched page:", n_rebuilt, "of", len(any_rows))
print("DiffSTE pages patched (box pasted into the rebuilt page):", n_patched, "of", len(any_rows))
# the page ids that needed a rebuild (first 20 only)
print("rebuilt page ids:", [p for p, v in rebuilt_pages.items() if v][:20])
# save the statistics next to the work files
stats.to_csv(os.path.join(WORK_DIR, "outside_box_before_rebuild.csv"), index=False)

## 8. Masks, one box per page, and the final checks

**Why (ELI5):** every page gets one white rectangle mask (white = edited box), shared by all versions of that page.
We then prove three things: each mask has exactly ONE white box that is a clean rectangle; the box matches the plan; and
after the rebuild, **zero** pixels differ outside the box between each AI page and its untouched page.

In [ ]:
# the checks below collect problems here
problems = {"blobs": [], "not_rect": [], "mask_size": [], "plan_vs_mask": [], "ai_size": [], "outside": []}
# one row per page for the mask / box description
mask_rows = []
# go through all page rows of all three sets
for r in pages.to_dict("records"):
    # which mask folder: TextCtrl has its own, the shared pages share one
    mask_set = "textctrl" if r["set"] == "textctrl" else "shared49"
    # where the final mask goes
    mask_dir = os.path.join(OUT_DIR, "masks", "box", mask_set)
    # make sure the folder exists
    os.makedirs(mask_dir, exist_ok=True)
    # the mask file
    mask_out = os.path.join(mask_dir, r["page"] + ".png")
    # describe the SOURCE mask (white blobs, rectangle, size)
    b = box_from_mask(r["mask_src"])
    # exactly one white blob?
    if b["blobs"] != 1:
        problems["blobs"].append(r["set"] + "/" + r["page"] + " blobs=" + str(b["blobs"]))
    # a clean rectangle?
    if not b["is_rectangle"]:
        problems["not_rect"].append(r["set"] + "/" + r["page"])
    # mask size must equal the page size
    if (b["mask_w"], b["mask_h"]) != (r["W"], r["H"]):
        problems["mask_size"].append(r["set"] + "/" + r["page"] + " mask " + str((b["mask_w"], b["mask_h"])) + " page " + str((r["W"], r["H"])))
    # TextCtrl: the plan box must be the same as the mask box
    if r["set"] == "textctrl" and (b["x1"], b["y1"], b["x2"], b["y2"]) != (r["x1"], r["y1"], r["x2"], r["y2"]):
        problems["plan_vs_mask"].append(r["page"] + " plan " + str((r["x1"], r["y1"], r["x2"], r["y2"])) + " mask " + str((b["x1"], b["y1"], b["x2"], b["y2"])))
    # write the clean rectangle mask (255 inside the box) once per shared page and once per TextCtrl page
    if not os.path.exists(mask_out):
        # a black page
        m = np.zeros((int(r["H"]), int(r["W"])), dtype=np.uint8)
        # white box
        m[r["y1"]:r["y2"], r["x1"]:r["x2"]] = 255
        # save as 8-bit greyscale PNG
        Image.fromarray(m).save(mask_out)
    # the final AI and untouched files on disk
    untouched_set = "textctrl" if r["set"] == "textctrl" else "shared49"
    # their paths
    p_unt = image_path(untouched_set, "untouched", r["page"])
    p_ai = image_path(r["set"], "ai_edit", r["page"])
    # re-read both from disk (this is what the detectors will see)
    u, a = np.array(Image.open(p_unt).convert("RGB")), np.array(Image.open(p_ai).convert("RGB"))
    # sizes must match
    if u.shape != a.shape or u.shape[1] != r["W"] or u.shape[0] != r["H"]:
        problems["ai_size"].append(r["set"] + "/" + r["page"])
        continue
    # pixels that differ outside the box
    n, _, _ = outside_diff(u, a, r)
    # must be zero
    if n != 0:
        problems["outside"].append(r["set"] + "/" + r["page"] + " " + str(n))

# turn problems into hard checks
check("every page has exactly one white box", str(len(problems["blobs"])) + " bad " + str(problems["blobs"][:5]), len(problems["blobs"]) == 0)
# rectangle check
check("every white box is a clean rectangle", str(len(problems["not_rect"])) + " bad " + str(problems["not_rect"][:5]), len(problems["not_rect"]) == 0)
# mask size
check("mask size = page size", str(len(problems["mask_size"])) + " bad " + str(problems["mask_size"][:3]), len(problems["mask_size"]) == 0)
# TextCtrl plan box
check("TextCtrl plan box = mask box", str(len(problems["plan_vs_mask"])) + " bad " + str(problems["plan_vs_mask"][:3]), len(problems["plan_vs_mask"]) == 0)
# AI output size = untouched size
check("AI output same size as untouched page", str(len(problems["ai_size"])) + " bad " + str(problems["ai_size"][:5]), len(problems["ai_size"]) == 0)
# the key check: nothing differs outside the box
check("pixels changed outside the box (untouched vs ai_edit, after rebuild)", str(len(problems["outside"])) + " pages differ " + str(problems["outside"][:5]), len(problems["outside"]) == 0)
# information: how many pages were rebuilt
check("shared pages with rebuilt untouched page", str(n_rebuilt) + " of " + str(len(any_rows)), True, hard=False)
# information: DiffSTE patched
check("DiffSTE pages patched", str(n_patched) + " of " + str(len(any_rows)), True, hard=False)
# write the checks so far
pd.DataFrame(CHECKS).to_csv(os.path.join(OUT_DIR, "checks.csv"), index=False)
# show all checks in one table
print(pd.DataFrame(CHECKS)[["check", "value", "passed", "hard"]].to_string(index=False))
# stop here if any hard check failed
stop_if_failed()
print("N1-a: ALL HARD CHECKS PASSED. Continue below with N1-b (generation, copy-move, splicing, JPEG).")

## 9. Generation helpers (reused from Week 7 notebook 06) and the JPEG helpers

**Why (ELI5):** *generation* is how DocTamper makes a fake the old-fashioned way: wipe out the old word, print a new
one in a similar font. We reuse notebook 06's recipe unchanged (Otsu finds the ink, grow it by 1 px, Telea inpainting
fills the hole, PIL prints the new text at the old ink's height and colour).

**One change:** notebook 06 did "decide + draw" in one function. Here the *decision* (which pixels are ink, font size,
colour, where to draw) is made ONCE on the PNG page and the same decision is *applied* to the JPEG page. So PNG and
JPEG fakes differ only by compression.

**JPEG pipeline (same for all five versions):** save PNG page as JPEG q90 → decode → make the edit → save as JPEG q90.
`untouched` goes through the same two saves with an "empty edit", so nothing but the edit differs between versions.

In [ ]:
# OpenCV: thresholds and inpainting
import cv2
# PIL: drawing text with a font
from PIL import ImageDraw, ImageFont

# square used to grow the text pixels by OTSU_DILATE px in every direction
GROW = np.ones((2 * OTSU_DILATE + 1, 2 * OTSU_DILATE + 1), np.uint8)

# ask fontconfig for the installed font files (empty list when fc-list does not exist)
try:
    # one font path per line
    import subprocess
    font_files = subprocess.run(["fc-list", "--format", "%{file}\n"], capture_output=True, text=True).stdout.splitlines()
# fc-list missing
except FileNotFoundError:
    # no list
    font_files = []
# pick the first preferred font that is installed (same order as notebook 06)
FONT_PATH = None
# Liberation Sans first, then DejaVu Sans
for wanted in ["LiberationSans-Regular.ttf", "DejaVuSans.ttf"]:
    # installed files with exactly this name
    hits = sorted(p for p in font_files if os.path.basename(p) == wanted)
    # found: use it
    if hits:
        # first match
        FONT_PATH = hits[0]
        # stop looking
        break
# fallback: the DejaVu Sans font that ships with matplotlib
if FONT_PATH is None:
    # matplotlib's font finder
    import matplotlib.font_manager
    # path of its DejaVu Sans file
    FONT_PATH = matplotlib.font_manager.findfont("DejaVu Sans")
# show and record the font
print("Font used:", FONT_PATH)
# add the font to config.json
config["font"] = FONT_PATH
# re-save config.json
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)


# find the ink pixels in a grey crop with Otsu (1 = ink, 0 = paper); ink = the smaller group
def find_text_pixels(grey):
    # Otsu picks the cut-off; darker pixels become 1
    _, dark = cv2.threshold(grey, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # if "dark" covers more than half the crop, the ink is the light part
    inverted = dark.mean() > 0.5
    # light ink: swap
    if inverted:
        # swap
        dark = 1 - dark
    # give back the 0/1 map and the flag
    return dark.astype(np.uint8), bool(inverted)


# the biggest font whose ink is at most max_h tall and max_w wide
def fit_font(text, max_h, max_w):
    # start from a tiny font in case nothing fits
    best = ImageFont.truetype(FONT_PATH, 4)
    # try bigger and bigger sizes
    for size in range(4, 300):
        # load the font at this size
        font = ImageFont.truetype(FONT_PATH, size)
        # ink box of the text: left, top, right, bottom
        left, top, right, bottom = font.getbbox(text)
        # too tall or too wide: keep the previous size
        if bottom - top > max_h or right - left > max_w:
            break
        # it fits: keep it
        best = font
    # the winner
    return best


# DECIDE a generation edit on one box crop (RGB): which pixels to erase, font size, colour, where to draw
def decide_generation(crop, target):
    # grey version for Otsu
    grey = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    # 1 = old ink pixel
    text, inverted = find_text_pixels(grey)
    # rows and columns of all ink pixels
    ys, xs = np.nonzero(text)
    # normal case: ink box and median colour of the old text
    if len(ys) > 0:
        # top row, bottom row and left column of the old ink
        top, bottom, left = int(ys.min()), int(ys.max()), int(xs.min())
        # median red, green, blue of the old ink
        colour = tuple(int(v) for v in np.median(crop[text == 1], axis=0))
    # no ink found (blank box): use the whole box height and black ink
    else:
        # whole box height, from the left edge
        top, bottom, left = 0, crop.shape[0] - 1, 0
        # black ink
        colour = (0, 0, 0)
    # grow the ink so anti-aliased edges are erased too
    erase = cv2.dilate(text, GROW)
    # ink height of the old text
    text_h = int(bottom - top + 1)
    # biggest font that fits between the old left edge and the box edge
    font = fit_font(target, max_h=text_h, max_w=crop.shape[1] - left)
    # gap between where PIL starts drawing and where the ink starts
    ink_left, ink_top, _, _ = font.getbbox(target)
    # everything we decided
    return dict(erase=erase, font_size=int(font.size), colour=colour, xy=(left - ink_left, top - ink_top),
                text_h=text_h, inverted=inverted, text_pixels=int(text.sum()), target=target)


# APPLY a decided generation edit to a box crop of any version of the page (PNG or JPEG pixels)
def apply_generation(crop, d):
    # fill the erased pixels from their surroundings (Telea)
    clean = cv2.inpaint(np.ascontiguousarray(crop), d["erase"], INPAINT_RADIUS, cv2.INPAINT_TELEA)
    # open the cleaned crop for drawing
    pil = Image.fromarray(clean)
    # the font at the decided size
    font = ImageFont.truetype(FONT_PATH, d["font_size"])
    # draw the new text at the decided place in the decided colour
    ImageDraw.Draw(pil).text(d["xy"], d["target"], font=font, fill=d["colour"])
    # back to numbers
    return np.array(pil)


# read a PNG as an RGB array
def load_png(path):
    # open and convert
    return np.array(Image.open(path).convert("RGB"))


# save an RGB array as JPEG quality JPEG_QUALITY and decode it again (one "save")
def jpeg_once(arr):
    # an in-memory file
    buf = io.BytesIO()
    # JPEG save
    Image.fromarray(arr).save(buf, format="JPEG", quality=JPEG_QUALITY)
    # back to the start
    buf.seek(0)
    # decode
    return np.array(Image.open(buf).convert("RGB"))


# write an RGB array to disk as JPEG quality JPEG_QUALITY (this is the second save)
def save_jpg(arr, path):
    # JPEG save
    Image.fromarray(arr).save(path, format="JPEG", quality=JPEG_QUALITY)


# the units we build: TextCtrl pages, and the shared pages (AnyText and DiffSTE use the same untouched page)
units = []
# TextCtrl pages
for r in pages[pages["set"] == "textctrl"].to_dict("records"):
    # one unit per page
    units.append(dict(set="textctrl", page=r["page"], editors=["textctrl"], row=r))
# shared pages (use the anytext rows; the boxes and texts are identical for diffste)
for r in pages[pages["set"] == "anytext"].to_dict("records"):
    # one unit per page
    units.append(dict(set="shared49", page=r["page"], editors=["anytext", "diffste"], row=r))

# demo on the first unit: decide, apply, and show that only the box changed
u0 = units[0]
# its box
r0 = u0["row"]
# its untouched page
U0 = load_png(image_path(u0["set"], "untouched", u0["page"]))
# decide on the box crop
d0 = decide_generation(U0[r0["y1"]:r0["y2"], r0["x1"]:r0["x2"]].copy(), r0["new_text"])
# the faked box
new0 = apply_generation(U0[r0["y1"]:r0["y2"], r0["x1"]:r0["x2"]].copy(), d0)
# show the decision without the big erase mask
print(u0["page"], {k: v for k, v in d0.items() if k != "erase"})
# the new crop must have the same shape as the box
print("new crop shape", new0.shape, "box shape", (r0["y2"] - r0["y1"], r0["x2"] - r0["x1"]))
print("units:", len(units), "| textctrl", sum(u["set"] == "textctrl" for u in units), "| shared49", sum(u["set"] == "shared49" for u in units))

## 10. Read every untouched page with EasyOCR (the donor word list)

**Why (ELI5):** copy-move and splicing paste *text that already exists*. To find such text we read every untouched page
with EasyOCR, which gives, per word: the text, how sure it is, and a box. For each word we also measure its **ink
height** (Otsu, same as for the box we are editing) and its **ink colour** (median colour of the ink pixels).

We keep only words EasyOCR is at least 70% sure of (the same rule as Week 4), with a few ink pixels.
Reading pages is the slow step; results are saved per page so a re-run skips finished pages.

In [ ]:
# load EasyOCR (install it if missing)
try:
    # import
    import easyocr
# not installed yet
except ImportError:
    # pip from python
    import subprocess, sys
    # install quietly
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "easyocr"])
    # import now
    import easyocr
# use the GPU when there is one
try:
    # torch tells us
    import torch
    # True on Kaggle GPU
    use_gpu = torch.cuda.is_available()
# torch missing
except Exception:
    # CPU only
    use_gpu = False
# English reader, same settings as Week 4 (the allowlist is chosen per call)
reader = easyocr.Reader(["en"], gpu=use_gpu)
# say which one we got
print("EasyOCR on GPU:", use_gpu)

# folder for the per-page word lists (cache, not zipped)
WORDS_DIR = os.path.join(WORK_DIR, "words")
# create it
os.makedirs(WORDS_DIR, exist_ok=True)
# the columns of a word list
WORD_COLS = ["set", "page", "text", "conf", "x1", "y1", "x2", "y2", "ink_h", "cr", "cg", "cb", "W", "H"]

# read all words on one page array
def read_words(set_name, page, arr):
    # page size
    H, W = arr.shape[:2]
    # no shrinking: let EasyOCR work at the page's own size
    found = reader.readtext(arr, canvas_size=int(max(H, W)))
    # the rows we keep
    out = []
    # each detection: 4 corner points, text, confidence
    for corners, text, conf in found:
        # axis-aligned box around the corners, clipped to the page
        xs = [int(p[0]) for p in corners]
        ys = [int(p[1]) for p in corners]
        # clip
        x1, x2, y1, y2 = max(min(xs), 0), min(max(xs), W), max(min(ys), 0), min(max(ys), H)
        # empty box: skip
        if x2 <= x1 or y2 <= y1:
            continue
        # the crop under the box
        crop = arr[y1:y2, x1:x2]
        # find its ink
        ink, _ = find_text_pixels(cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY))
        # rows of ink pixels
        iy, ix = np.nonzero(ink)
        # no ink: skip
        if len(iy) == 0:
            continue
        # ink height and median ink colour
        ink_h = int(iy.max() - iy.min() + 1)
        # colour
        col = np.median(crop[ink == 1], axis=0)
        # one row
        out.append(dict(set=set_name, page=page, text=str(text), conf=float(conf), x1=x1, y1=y1, x2=x2, y2=y2,
                        ink_h=ink_h, cr=float(col[0]), cg=float(col[1]), cb=float(col[2]), W=W, H=H))
    # a table (possibly empty, with all columns)
    return pd.DataFrame(out, columns=WORD_COLS)

# stopwatch
t_ocr = time.time()
# collect all pages' words
all_words = []
# go unit by unit
for i, u in enumerate(units):
    # cache file of this page
    cache = os.path.join(WORDS_DIR, u["set"] + "_" + u["page"] + ".csv")
    # cached: read it
    if os.path.exists(cache):
        # read (text as text)
        w = pd.read_csv(cache, dtype={"text": str}, keep_default_na=False)
    # not cached: run OCR on the untouched page and save
    else:
        # read the page
        w = read_words(u["set"], u["page"], load_png(image_path(u["set"], "untouched", u["page"])))
        # save
        w.to_csv(cache, index=False)
    # keep
    all_words.append(w)
    # progress line every 20 pages
    if (i + 1) % 20 == 0:
        print("  OCR", i + 1, "of", len(units), "pages,", int(time.time() - t_ocr), "s")
# one big table
words = pd.concat(all_words, ignore_index=True)
# the kind of each word by the same rule as the edits
words["kind"] = [text_kind(t) for t in words["text"]]
# text without spaces, lower case (for "different text" tests)
words["text_norm"] = [str(t).replace(" ", "").lower() for t in words["text"]]
# a word is a valid donor if EasyOCR is sure enough and the ink is at least 6 px tall
words["valid"] = (words["conf"] >= OCR_MIN_CONF) & (words["ink_h"] >= 6) & (words["text_norm"] != "")
# summary per set
print("pages read:", len(units), "| words found:", len(words), "| time", int(time.time() - t_ocr), "s")
# valid words and kinds per set
print(words[words["valid"]].groupby(["set", "kind"]).size().unstack(fill_value=0).to_string())
# save the word table (work file)
words.to_csv(os.path.join(WORK_DIR, "words_all.csv"), index=False)

## 11. Decide every edit ONCE (generation, copy-move donor, splicing donor)

**Why (ELI5):** a fake must change only the method, so every choice is made here, once, and then used for both PNG and
JPEG. For each page:

- **generation:** same new text as the AI edit (decided on the PNG crop, section 9).
- **copy-move:** a donor word from the SAME page. **splicing:** a donor word from ANOTHER page of the same set.

**Donor rules (CLAUDE.md):** same kind as the new text (number for number, word for word); different text from the new
text and from the old text; ink height within 10% of the old text's ink height, else within 20% (that is the `tier`);
ink colour within `COLOUR_TOL` (RGB distance). Copy-move donors must not overlap the box. If several donors are valid, a
seeded random choice picks one. **No donor → the page is skipped for that version and logged** (`skipped=True`).

The pasted patch is a rectangle the size of the box, centred on the donor word (moved inwards if it would leave the
page), copied as-is with nothing erased.

In [ ]:
# the donor word boxes overlap the edit box?
def overlaps(w, r):
    # two rectangles overlap unless one is entirely left/right/above/below the other
    return not (w["x2"] <= r["x1"] or w["x1"] >= r["x2"] or w["y2"] <= r["y1"] or w["y1"] >= r["y2"])

# pick a donor for one unit and one version ("copy_move" or "splicing"); None if there is no valid donor
def choose_donor(version, u):
    # the page's row (box, text, kind)
    r = u["row"]
    # the generation decision holds the old text's ink height and colour
    g = u["gen"]
    # the box size
    bw, bh = int(r["x2"] - r["x1"]), int(r["y2"] - r["y1"])
    # only valid words of this set
    pool = words[(words["set"] == u["set"]) & words["valid"]]
    # copy-move: same page; splicing: other pages
    if version == "copy_move":
        # same page
        pool = pool[pool["page"] == u["page"]]
        # not overlapping the edit box
        keep = pd.Series([not overlaps(w, r) for w in pool.to_dict("records")], index=pool.index, dtype=bool)
        # keep only the words that do not overlap
        pool = pool[keep]
    # splicing
    else:
        # other pages of the same set
        pool = pool[pool["page"] != u["page"]]
    # same kind as the new text
    pool = pool[pool["kind"] == r["kind"]]
    # text must differ from the new text and from the old text
    banned = {str(r["new_text"]).replace(" ", "").lower(), str(r["source_text"]).replace(" ", "").lower()}
    # remove them
    pool = pool[~pool["text_norm"].isin(banned)]
    # the donor page must be big enough for the patch
    pool = pool[(pool["W"] >= bw) & (pool["H"] >= bh)]
    # ink colour distance to the old text's ink colour
    dist = np.sqrt((pool["cr"] - g["colour"][0]) ** 2 + (pool["cg"] - g["colour"][1]) ** 2 + (pool["cb"] - g["colour"][2]) ** 2)
    # add as a column
    pool = pool.assign(colour_dist=dist, height_ratio=pool["ink_h"] / g["text_h"])
    # colour must be close enough
    pool = pool[pool["colour_dist"] <= COLOUR_TOL]
    # try the 10% tier, then the 20% tier
    for tier, tol in HEIGHT_TIERS:
        # donors inside this height tolerance
        ok = pool[(pool["height_ratio"] - 1).abs() <= tol]
        # none: next tier
        if len(ok) == 0:
            continue
        # fixed order so the seeded choice is repeatable
        ok = ok.sort_values(["page", "y1", "x1"]).reset_index(drop=True)
        # a random generator seeded by run, version and page
        rng = random.Random(str(SEED) + "|" + version + "|" + u["set"] + "|" + u["page"])
        # the chosen donor
        w = ok.iloc[rng.randrange(len(ok))]
        # patch top-left: centred on the donor word, moved inwards if needed
        px = int(min(max((w["x1"] + w["x2"]) // 2 - bw // 2, 0), w["W"] - bw))
        # same vertically
        py = int(min(max((w["y1"] + w["y2"]) // 2 - bh // 2, 0), w["H"] - bh))
        # everything we need later
        return dict(donor_page=w["page"], donor_text=w["text"], donor_box="%d,%d,%d,%d" % (w["x1"], w["y1"], w["x2"], w["y2"]),
                    height_ratio=round(float(w["height_ratio"]), 3), colour_dist=round(float(w["colour_dist"]), 1),
                    tier=tier, patch_xy=(px, py), n_valid=int(len(ok)))
    # no donor in either tier
    return None

# one row per decision, for donors.csv
donor_rows = []
# go unit by unit
for u in units:
    # the page's row
    r = u["row"]
    # decide the generation edit on the untouched box crop
    U = load_png(image_path(u["set"], "untouched", u["page"]))
    # decision
    u["gen"] = decide_generation(U[r["y1"]:r["y2"], r["x1"]:r["x2"]].copy(), r["new_text"])
    # the two donor versions
    for version in ["copy_move", "splicing"]:
        # choose
        d = choose_donor(version, u)
        # remember (None = skipped)
        u[version] = d
        # donors.csv row
        if d is None:
            # skipped page
            donor_rows.append(dict(version=version, set=u["set"], page=u["page"], donor_page="", donor_text="", donor_box="",
                                   height_ratio="", colour_dist="", tier="", skipped=True, kind=r["kind"], n_valid=0, patch_xy=""))
        # donor found
        else:
            # keep the row (patch_xy as text)
            donor_rows.append(dict(version=version, set=u["set"], page=u["page"], donor_page=d["donor_page"], donor_text=d["donor_text"],
                                   donor_box=d["donor_box"], height_ratio=d["height_ratio"], colour_dist=d["colour_dist"], tier=d["tier"],
                                   skipped=False, kind=r["kind"], n_valid=d["n_valid"], patch_xy="%d,%d" % d["patch_xy"]))
# make the table and save it into the dataset
donors = pd.DataFrame(donor_rows)
# write donors.csv
donors.to_csv(os.path.join(OUT_DIR, "donors.csv"), index=False)
# summary: per version and set: pages, skipped, tier counts
print(donors.groupby(["version", "set"]).agg(pages=("page", "count"), skipped=("skipped", "sum")).to_string())
# tier counts
print(donors[~donors["skipped"]].groupby(["version", "tier"]).size().to_string())
# skipped pages by kind (first 10 rows only)
print(donors[donors["skipped"]][["version", "set", "page", "kind"]].head(10).to_string(index=False))
# a few donor examples (text only)
print(donors[~donors["skipped"]][["version", "page", "kind", "donor_page", "donor_text", "height_ratio", "colour_dist", "tier"]].head(6).to_string(index=False))

## 12. Make every version, PNG and JPEG

**Why (ELI5):** now we cook. For each page we build five versions and save each twice (PNG and JPEG), always starting
from the SAME untouched page, so the only things that differ are the edit and the compression.

| version | PNG page | JPEG page |
| --- | --- | --- |
| untouched | the untouched page | save q90 → decode → save q90 |
| ai_edit | box pixels of the AI output pasted into the untouched page (done in section 7) | same box pixels pasted into the once-saved JPEG page → save q90 |
| generation | decided edit applied to the box | same decision applied to the once-saved JPEG page → save q90 |
| copy_move / splicing | donor patch pasted into the box | same patch (cut from the donor's once-saved JPEG page) pasted into the once-saved JPEG page → save q90 |

In [ ]:
# cache of the last few donor pages (PNG array and once-saved JPEG array), so splicing does not reload constantly
donor_cache = {}

# get a donor page's arrays
def donor_arrays(set_name, page):
    # the cache key
    key = (set_name, page)
    # not cached yet
    if key not in donor_cache:
        # keep the cache small
        if len(donor_cache) >= 6:
            # drop the oldest
            donor_cache.pop(next(iter(donor_cache)))
        # load the page
        arr = load_png(image_path(set_name, "untouched", page))
        # PNG array and once-saved JPEG array
        donor_cache[key] = (arr, jpeg_once(arr))
    # return them
    return donor_cache[key]

# the files a unit should have when it is finished (used to skip finished pages)
def expected_files(u):
    # the untouched JPEG
    files = [image_path(u["set"], "untouched", u["page"], "jpg")]
    # generation, both formats
    files += [image_path(u["set"], "generation", u["page"], f) for f in ["png", "jpg"]]
    # donor versions that were not skipped
    for version in ["copy_move", "splicing"]:
        # only if a donor exists
        if u[version] is not None:
            # both formats
            files += [image_path(u["set"], version, u["page"], f) for f in ["png", "jpg"]]
    # AI edits (png done in section 7, jpg now)
    files += [image_path(e, "ai_edit", u["page"], "jpg") for e in u["editors"]]
    # the list
    return files

# stopwatch and counters
t_make = time.time()
# files written in this run
n_written = 0
# go unit by unit
for i, u in enumerate(units):
    # skip finished pages (resumable)
    if all(os.path.exists(p) for p in expected_files(u)):
        continue
    # the box
    r = u["row"]
    # box as slices and size
    ys, xs = slice(int(r["y1"]), int(r["y2"])), slice(int(r["x1"]), int(r["x2"]))
    # box size
    bh, bw = int(r["y2"] - r["y1"]), int(r["x2"] - r["x1"])
    # untouched PNG page and the once-saved JPEG page
    U = load_png(image_path(u["set"], "untouched", u["page"]))
    # once-saved JPEG
    J1 = jpeg_once(U)
    # untouched JPEG = second save of the once-saved page
    save_jpg(J1, image_path(u["set"], "untouched", u["page"], "jpg"))
    # AI edits: paste the editor's box pixels into the once-saved JPEG page
    for e in u["editors"]:
        # the editor's final PNG page (from section 7)
        A = load_png(image_path(e, "ai_edit", u["page"]))
        # copy the JPEG page
        Aj = J1.copy()
        # paste the editor's box
        Aj[ys, xs] = A[ys, xs]
        # second save
        save_jpg(Aj, image_path(e, "ai_edit", u["page"], "jpg"))
    # generation: same decision on both pages
    g_png, g_jpg = U.copy(), J1.copy()
    # apply to the PNG box
    g_png[ys, xs] = apply_generation(U[ys, xs].copy(), u["gen"])
    # apply to the JPEG box
    g_jpg[ys, xs] = apply_generation(J1[ys, xs].copy(), u["gen"])
    # save both
    Image.fromarray(g_png).save(image_path(u["set"], "generation", u["page"], "png"))
    # JPEG
    save_jpg(g_jpg, image_path(u["set"], "generation", u["page"], "jpg"))
    # copy-move and splicing
    for version in ["copy_move", "splicing"]:
        # the decision (None = skipped)
        d = u[version]
        # nothing to make
        if d is None:
            continue
        # the donor page arrays (the same page for copy-move)
        dU, dJ = donor_arrays(u["set"], d["donor_page"])
        # the patch corner
        px, py = d["patch_xy"]
        # paste the donor patch into the box of the PNG page
        p_png = U.copy()
        # paste (pure copy-paste, nothing erased)
        p_png[ys, xs] = dU[py:py + bh, px:px + bw]
        # same patch into the JPEG page
        p_jpg = J1.copy()
        # paste from the donor's once-saved JPEG page
        p_jpg[ys, xs] = dJ[py:py + bh, px:px + bw]
        # save both
        Image.fromarray(p_png).save(image_path(u["set"], version, u["page"], "png"))
        # JPEG
        save_jpg(p_jpg, image_path(u["set"], version, u["page"], "jpg"))
    # count this page as made
    n_written += 1
    # progress line every 20 pages
    if n_written % 20 == 0:
        print("  made", n_written, "pages,", int(time.time() - t_make), "s")
# how many files exist in the image folders now
n_png = len(glob.glob(os.path.join(OUT_DIR, "images", "*", "*", "png", "*.png")))
# JPEG files
n_jpg = len(glob.glob(os.path.join(OUT_DIR, "images", "*", "*", "jpg", "*.jpg")))
# report
print("pages made now:", n_written, "of", len(units), "| PNG files:", n_png, "| JPEG files:", n_jpg, "|", int(time.time() - t_make), "s")

## 13. Edit quality of the AI edits (EasyOCR)

**Why (ELI5):** did the AI really write the word we asked for? We let EasyOCR read the edited box and compare with the
intended text. Two numbers (same as Week 4):

- `ocr_correct`: after removing spaces and ignoring capitals, OCR text == intended text.
- `ned` = normalised edit distance = (fewest single-character fixes between the two texts) ÷ (length of the longer
  text). Example: OCR `Slaw` vs intended `Slow` → 1 fix ÷ 4 = 0.25. 0 = perfect, 1 = completely different.

**Reader settings:** English, same as Week 4, with an allowlist. TextCtrl edits are letters and digits only (Week 4
allowlist). AnyText/DiffSTE edits also contain `% & ( ) - . / : + ,`, so these characters are added to their allowlist;
otherwise the reader could never get them right.

In [ ]:
# Week 4's allowlist: letters and digits
ALLOWED_W4 = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
# AnyText/DiffSTE also have punctuation
ALLOWED_PUNCT = ALLOWED_W4 + "%&()-./:+,"

# classic edit distance: fewest single-character insertions, deletions, changes to turn a into b
def edit_distance(a, b):
    # row of distances for the empty prefix of a
    prev = list(range(len(b) + 1))
    # one row per character of a
    for i, ca in enumerate(a, 1):
        # the first entry: delete i characters
        cur = [i]
        # each character of b
        for j, cb in enumerate(b, 1):
            # best of delete, insert, change (free if equal)
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        # next row
        prev = cur
    # distance between the full strings
    return prev[-1]

# normalised edit distance (0 when both are empty)
def ned(a, b):
    # the longer length
    n = max(len(a), len(b))
    # empty vs empty: identical
    return 0.0 if n == 0 else edit_distance(a, b) / n

# tiny example for the reader of this notebook
print("example ned('slaw','slow') =", ned("slaw", "slow"))

# cache file for the OCR results
QUAL_CSV = os.path.join(WORK_DIR, "ai_quality.csv")
# earlier results (resumable)
if os.path.exists(QUAL_CSV):
    # read them
    qual_done = pd.read_csv(QUAL_CSV, dtype={"ai_ocr_text": str}, keep_default_na=False)
# none yet
else:
    # empty table
    qual_done = pd.DataFrame(columns=["set", "page", "ai_ocr_text", "ai_ocr_correct", "ai_ned"])
# which (set, page) pairs are done
done_keys = set(zip(qual_done["set"], qual_done["page"]))
# new rows
qual_rows = []
# stopwatch
t_q = time.time()
# every page of every set
for r in pages.to_dict("records"):
    # already done
    if (r["set"], r["page"]) in done_keys:
        continue
    # the final AI edit page (PNG)
    A = load_png(image_path(r["set"], "ai_edit", r["page"]))
    # the box crop
    crop = A[r["y1"]:r["y2"], r["x1"]:r["x2"]]
    # allowlist per set
    allow = ALLOWED_W4 if r["set"] == "textctrl" else ALLOWED_PUNCT
    # read the crop and join the pieces without spaces
    read = "".join(p[1] for p in reader.readtext(crop, allowlist=allow)).replace(" ", "")
    # the intended text without spaces
    target = str(r["new_text"]).replace(" ", "")
    # store (compare without capitals)
    qual_rows.append(dict(set=r["set"], page=r["page"], ai_ocr_text=read, ai_ocr_correct=bool(read.lower() == target.lower()),
                          ai_ned=round(ned(read.lower(), target.lower()), 3)))
# all results
qual = pd.concat([qual_done, pd.DataFrame(qual_rows)], ignore_index=True)
# save the cache
qual.to_csv(QUAL_CSV, index=False)
# summary per set
print(qual.groupby("set").agg(pages=("page", "count"), ocr_correct_share=("ai_ocr_correct", "mean"), mean_ned=("ai_ned", "mean")).round(3).to_string())
print("OCR time", int(time.time() - t_q), "s")
# first rows: intended vs read
show = qual.merge(pages[["set", "page", "new_text"]], on=["set", "page"]).head(6)
# print
print(show[["set", "page", "new_text", "ai_ocr_text", "ai_ocr_correct", "ai_ned"]].to_string(index=False))

## 14. The manifest

**Why (ELI5):** the manifest is the table of contents of the dataset: one row per image, saying which set, page,
version and format it is, where the file is (path relative to the `w7_sets` folder, so it works on Kaggle, Colab and a
laptop), where its mask is, and the facts about the edit. The next notebooks read only this table.

- `anytext` and `diffste` list all five versions; the four non-AI versions point to the `shared49` files.
- `editor` = the set's editor, for every row of that set.
- Pages skipped for a donor version have no row for that version (both formats).
- `ai_*` columns are filled on `ai_edit` rows only.

In [ ]:
# the file extension of a format
EXT = {"png": "png", "jpg": "jpg"}
# which pages were skipped for which donor version: {(set, version): set of pages}
skipped = {}
# collect from the donors table
for row in donors[donors["skipped"]].to_dict("records"):
    # key
    skipped.setdefault((row["set"], row["version"]), set()).add(row["page"])
# the OCR quality as a lookup: (set, page) -> row
qual_lookup = {(q["set"], q["page"]): q for q in qual.to_dict("records")}
# one manifest row per image
man_rows = []
# go through every page row of the page table
for r in pages.to_dict("records"):
    # the set that holds the non-AI versions
    src_set = "textctrl" if r["set"] == "textctrl" else "shared49"
    # the mask lives in the same folder
    mask_rel = "masks/box/" + src_set + "/" + r["page"] + ".png"
    # was the shared untouched page rebuilt?
    rebuilt = bool(rebuilt_pages.get(r["page"], False)) if r["set"] != "textctrl" else False
    # all five versions
    for version in VERSIONS:
        # donor versions skipped for this page
        if r["page"] in skipped.get((src_set, version), set()):
            continue
        # both formats
        for fmt in ["png", "jpg"]:
            # AI edits live in the set's own folder, the rest in the shared/TextCtrl folder
            folder_set = r["set"] if version == "ai_edit" else src_set
            # relative path
            rel = "images/%s/%s/%s/%s.%s" % (folder_set, version, fmt, r["page"], EXT[fmt])
            # the quality numbers (AI edits only)
            q = qual_lookup.get((r["set"], r["page"])) if version == "ai_edit" else None
            # one row
            man_rows.append(dict(set=r["set"], page=r["page"], version=version, format=fmt, path=rel, mask_path=mask_rel,
                                 editor=r["editor"], kind=r["kind"], new_text=r["new_text"],
                                 x1=r["x1"], y1=r["y1"], x2=r["x2"], y2=r["y2"], box_share=round(float(r["box_share"]), 6),
                                 size_group=r["size_group"],
                                 ai_ocr_text=q["ai_ocr_text"] if q else "", ai_ocr_correct=q["ai_ocr_correct"] if q else "",
                                 ai_ned=q["ai_ned"] if q else "", untouched_rebuilt=rebuilt))
# make the table
manifest = pd.DataFrame(man_rows)
# save it into the dataset
manifest.to_csv(os.path.join(OUT_DIR, "manifest.csv"), index=False)
# summary: rows per set / version / format
print(manifest.groupby(["set", "version"]).size().unstack(fill_value=0).to_string())
print("manifest rows:", len(manifest), "| sets:", manifest["set"].nunique())

## 15. Final checks (every hard check must pass, otherwise the notebook stops)

**Why (ELI5):** we re-read every file from disk, exactly as the detectors will, and prove the dataset is what we
said it is.

- **PNG:** pixels changed outside the box (versus the untouched page) = **0** for every version.
- **JPEG:** a JPEG cuts the page into 16 × 16 blocks, so changing the box also re-codes the blocks around it. Changed
  pixels outside the box are therefore allowed **only within 20 px of the box** (the width of the neighbouring JPEG
  blocks); anything further away is a failure. This is the JPEG version of "nothing changes outside the box".
- **JPEG files** really are JPEG with the quality-90 quantisation tables.
- Every version really changes something inside the box (warning only), every manifest path exists, row counts match
  the donor skips.

In [ ]:
# the quantisation tables that a quality-90 JPEG must have (made by saving a dummy image the same way)
buf = io.BytesIO()
# save a dummy at JPEG_QUALITY
Image.new("RGB", (16, 16), (128, 128, 128)).save(buf, format="JPEG", quality=JPEG_QUALITY)
# back to start
buf.seek(0)
# read its tables
expected_q = {k: list(v) for k, v in Image.open(buf).quantization.items()}

# how far (in pixels) a changed pixel is from the box (0 = inside the box edge)
def distance_to_box(ys_, xs_, r):
    # horizontal distance beyond the box
    dx = np.maximum(np.maximum(r["x1"] - xs_, xs_ - (r["x2"] - 1)), 0)
    # vertical distance beyond the box
    dy = np.maximum(np.maximum(r["y1"] - ys_, ys_ - (r["y2"] - 1)), 0)
    # largest of the two
    return np.maximum(dx, dy)

# collectors
bad = dict(missing=[], not_jpeg=[], png_outside=[], jpg_far=[], size=[], unchanged=[], mask_missing=[])
# the farthest changed JPEG pixel seen
max_far = 0
# one file per path (the anytext/diffste rows share the shared49 files)
files = manifest.drop_duplicates("path")
# go through every file
for r in files.to_dict("records"):
    # the file on disk
    full = os.path.join(OUT_DIR, r["path"])
    # missing file
    if not os.path.exists(full):
        bad["missing"].append(r["path"])
        continue
    # missing mask
    if not os.path.exists(os.path.join(OUT_DIR, r["mask_path"])):
        bad["mask_missing"].append(r["mask_path"])
    # JPEG: must be a JPEG with quality-90 tables
    if r["format"] == "jpg":
        # open without decoding
        im = Image.open(full)
        # format and tables
        if im.format != "JPEG" or {k: list(v) for k, v in im.quantization.items()} != expected_q:
            bad["not_jpeg"].append(r["path"])
    # the untouched file of the same page and format (not needed for the untouched version itself)
    if r["version"] == "untouched":
        continue
    # build its path
    parts = r["path"].split("/")
    # same folder set, version replaced by untouched
    unt_full = os.path.join(OUT_DIR, parts[0], parts[1] if r["version"] != "ai_edit" else ("textctrl" if r["set"] == "textctrl" else "shared49"), "untouched", parts[3], parts[4])
    # read the untouched page (decoded pixels, PNG or JPEG)
    a = np.array(Image.open(unt_full).convert("RGB"))
    # read the version
    b = np.array(Image.open(full).convert("RGB"))
    # sizes must match
    if a.shape != b.shape:
        bad["size"].append(r["path"])
        continue
    # per-pixel largest colour difference
    diff = np.abs(a.astype(np.int16) - b.astype(np.int16)).max(axis=2)
    # was something changed inside the box?
    inside = diff[r["y1"]:r["y2"], r["x1"]:r["x2"]]
    # only the PNG pages are tested for "really changed" (JPEG noise would fool the test)
    if r["format"] == "png" and inside.max() == 0:
        bad["unchanged"].append(r["path"])
    # blank the box
    diff[r["y1"]:r["y2"], r["x1"]:r["x2"]] = 0
    # coordinates of changed pixels outside the box
    cy, cx = np.nonzero(diff)
    # PNG: none allowed
    if r["format"] == "png":
        # any change is a failure
        if len(cy) > 0:
            bad["png_outside"].append(r["path"] + " " + str(len(cy)))
    # JPEG: changes allowed only close to the box
    else:
        # some changes
        if len(cy) > 0:
            # the farthest one
            far = int(distance_to_box(cy, cx, r).max())
            # remember the worst
            max_far = max(max_far, far)
            # too far away
            if far > 20:
                bad["jpg_far"].append(r["path"] + " " + str(far) + "px")

# hard checks
check("every manifest path exists", str(len(bad["missing"])) + " missing " + str(bad["missing"][:3]), len(bad["missing"]) == 0)
# masks
check("every manifest mask exists", str(len(bad["mask_missing"])) + " missing", len(bad["mask_missing"]) == 0)
# JPEG quality
check("JPEG files are JPEG quality " + str(JPEG_QUALITY), str(len(bad["not_jpeg"])) + " bad " + str(bad["not_jpeg"][:3]), len(bad["not_jpeg"]) == 0)
# sizes
check("every version same size as its untouched page", str(len(bad["size"])) + " bad " + str(bad["size"][:3]), len(bad["size"]) == 0)
# PNG outside the box
check("PNG: pixels changed outside the box = 0 (all versions)", str(len(bad["png_outside"])) + " files differ " + str(bad["png_outside"][:3]), len(bad["png_outside"]) == 0)
# JPEG outside the box
check("JPEG: changed pixels outside the box only within 20 px", "bad " + str(len(bad["jpg_far"])) + ", farthest change " + str(max_far) + " px " + str(bad["jpg_far"][:3]), len(bad["jpg_far"]) == 0)
# every version changes the box (warning only)
check("PNG versions identical to untouched inside the box", str(len(bad["unchanged"])) + " " + str(bad["unchanged"][:3]), len(bad["unchanged"]) == 0, hard=False)
# donor skips: counted and consistent with the manifest
n_skip_cm = int(donors[(donors["version"] == "copy_move") & donors["skipped"]].shape[0])
# splicing skips
n_skip_sp = int(donors[(donors["version"] == "splicing") & donors["skipped"]].shape[0])
# expected rows: every page of every set x 5 versions x 2 formats, minus skipped pages (a shared skip counts for anytext AND diffste)
exp_rows = 0
# per page row
for r in pages.to_dict("records"):
    # the set that holds the non-AI versions
    src_set = "textctrl" if r["set"] == "textctrl" else "shared49"
    # versions made for this page
    made = [v for v in VERSIONS if r["page"] not in skipped.get((src_set, v), set())]
    # two formats each
    exp_rows += 2 * len(made)
# compare with the manifest
check("donor skips counted (copy_move %d, splicing %d unit pages) and manifest rows match" % (n_skip_cm, n_skip_sp),
      "manifest " + str(len(manifest)) + " rows, expected " + str(exp_rows), len(manifest) == exp_rows)
# the manifest has no duplicate images
check("no duplicate (set, page, version, format) in the manifest", str(int(manifest.duplicated(["set", "page", "version", "format"]).sum())) + " duplicates",
      int(manifest.duplicated(["set", "page", "version", "format"]).sum()) == 0)
# information: SMOKE flag
check("run mode", "SMOKE" if SMOKE else "FULL", True, hard=False)
# save all checks (N1-a and N1-b)
pd.DataFrame(CHECKS).to_csv(os.path.join(OUT_DIR, "checks.csv"), index=False)
# show all checks
print(pd.DataFrame(CHECKS)[["check", "value", "passed", "hard"]].to_string(index=False, max_colwidth=90))
# stop when any hard check failed
stop_if_failed()
print("ALL HARD CHECKS PASSED.")

## 16. QC figure (for us only, NOT for slides: it shows bank-statement pixels, NDA)

**Why (ELI5):** numbers can hide silly mistakes, so we look at a few boxes. 6 pages (2 per set), 5 versions each, crops of
the box only (PNG). Look for: ai_edit shows the new text; generation shows the same new text in a similar font;
copy_move / splicing show different existing text of the same kind; untouched shows the old text.

In [ ]:
# matplotlib for the figure
import matplotlib
# draw without a screen
matplotlib.use("Agg")
# plotting tools
import matplotlib.pyplot as plt
# folder for figures
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
# pick 2 pages per set with all five versions, seeded
picker = random.Random(SEED)
# the rows of the figure: (set, page)
picked = []
# per set
for s in ["textctrl", "anytext", "diffste"]:
    # pages of this set that have all five versions
    ok_pages = [p for p in pages[pages["set"] == s]["page"] if len(manifest[(manifest["set"] == s) & (manifest["page"] == p) & (manifest["format"] == "png")]) == 5]
    # choose two
    picked += [(s, p) for p in picker.sample(ok_pages, min(2, len(ok_pages)))]
# the figure: one row per page, one column per version
fig, axes = plt.subplots(len(picked), 5, figsize=(15, 1.4 * len(picked) + 0.8), squeeze=False)
# fill it
for i, (s, p) in enumerate(picked):
    # the page row
    r = manifest[(manifest["set"] == s) & (manifest["page"] == p)].iloc[0]
    # each version
    for j, version in enumerate(VERSIONS):
        # its PNG file
        row = manifest[(manifest["set"] == s) & (manifest["page"] == p) & (manifest["version"] == version) & (manifest["format"] == "png")].iloc[0]
        # the page and its box crop
        crop = load_png(os.path.join(OUT_DIR, row["path"]))[r["y1"]:r["y2"], r["x1"]:r["x2"]]
        # draw
        axes[i][j].imshow(crop, interpolation="nearest")
        # no ticks
        axes[i][j].set_xticks([])
        # no ticks
        axes[i][j].set_yticks([])
        # column titles on the first row
        if i == 0:
            axes[i][j].set_title(version)
    # row label: set and page
    axes[i][0].set_ylabel(s + "\n" + p, fontsize=8)
# tidy and save
plt.tight_layout()
# save the figure into the dataset
plt.savefig(os.path.join(OUT_DIR, "figures", "qc_versions.png"), dpi=110)
# show it in the notebook
plt.show()
print("saved figures/qc_versions.png")

## 17. Three zips

**Why (ELI5):** one folder, three different jobs, so three zips:

1. **ALL outputs** — everything this notebook wrote (the dataset + work files). Your own backup.
2. **SAFE for Claude (no NDA content)** — tables only: no page images, no crops, no QC figure, and the text columns
   (`new_text`, `ai_ocr_text`, `donor_text`) are removed, because the words come from real bank statements. Page ids,
   boxes, kinds, size groups, OCR correctness, donor ratios and all checks are kept. Safe to paste into Claude.
3. **NEXT ROUND** — only what notebooks 2A, 2B, 2C and 3 read: `images/`, `masks/`, `manifest.csv`, `config.json`.
   Upload it as the Kaggle dataset `PATH_TO_W7_COMMON_SETS_DATASET` (Kaggle unzips it). **Nazhan uses Colab, not Kaggle:** send him this
   zip (Google Drive or Colab upload), unzip it so the `w7_sets/` folder holds `manifest.csv`; all paths in the manifest
   are relative to that folder.

In [ ]:
# make a zip from a list of (file on disk, name inside the zip)
def make_zip(zip_path, items):
    # images are already compressed, so store them; text files are deflated
    with zipfile.ZipFile(zip_path, "w") as z:
        # add each file
        for src, arc in items:
            # the compression method
            method = zipfile.ZIP_STORED if src.lower().endswith((".png", ".jpg")) else zipfile.ZIP_DEFLATED
            # write
            z.write(src, arc, compress_type=method)
    # size in megabytes
    return os.path.getsize(zip_path) / 1e6

# every file under a folder as (path, name inside zip), names starting with `prefix`
def walk_items(folder, prefix, skip_dirs=()):
    # collected files
    items = []
    # walk the folder
    for root, dirs, fs in os.walk(folder):
        # leave out folders we do not want
        dirs[:] = [d for d in dirs if d not in skip_dirs]
        # add every file
        for f in fs:
            # full path
            full = os.path.join(root, f)
            # name inside the zip
            items.append((full, os.path.join(prefix, os.path.relpath(full, folder))))
    # the list
    return items

# ---------- (2) the NDA-safe tables, written to a small folder ----------
SAFE_DIR = os.path.join(WORK_DIR, "safe_tables")
# create it
os.makedirs(SAFE_DIR, exist_ok=True)
# manifest without the text columns
manifest.drop(columns=["new_text", "ai_ocr_text"]).to_csv(os.path.join(SAFE_DIR, "manifest_safe.csv"), index=False)
# donors without the donor text and box position on the donor page
donors.drop(columns=["donor_text", "donor_box", "patch_xy"]).to_csv(os.path.join(SAFE_DIR, "donors_safe.csv"), index=False)
# the page-level OCR quality without any text
qual.drop(columns=["ai_ocr_text"]).to_csv(os.path.join(SAFE_DIR, "ai_quality_safe.csv"), index=False)
# the outside-the-box statistics before the rebuild (numbers only)
stats.to_csv(os.path.join(SAFE_DIR, "outside_box_before_rebuild.csv"), index=False)
# a short explanation inside the zip
with open(os.path.join(SAFE_DIR, "README.txt"), "w") as f:
    # what is inside and what was removed
    f.write("Safe to share: tables only. Removed: page images, crops, QC figure, and the text columns new_text, ai_ocr_text, donor_text.\n")

# ---------- build the three zips ----------
# (1) everything: the dataset folder and the work files (without the big caches)
all_items = walk_items(OUT_DIR, "w7_sets") + walk_items(WORK_DIR, "w7_work", skip_dirs=("untouched_render",))
# (2) safe tables + checks + config
safe_items = walk_items(SAFE_DIR, "w7_safe") + [(os.path.join(OUT_DIR, "checks.csv"), "w7_safe/checks.csv"), (os.path.join(OUT_DIR, "config.json"), "w7_safe/config.json")]
# (3) what the next notebooks need
next_items = walk_items(os.path.join(OUT_DIR, "images"), "w7_sets/images") + walk_items(os.path.join(OUT_DIR, "masks"), "w7_sets/masks") \
    + [(os.path.join(OUT_DIR, "manifest.csv"), "w7_sets/manifest.csv"), (os.path.join(OUT_DIR, "config.json"), "w7_sets/config.json")]
# the three zip files
ZIPS = {"1_ALL_outputs": os.path.join(WORKING, "w7_1_ALL_outputs.zip"),
        "2_SAFE_for_claude_no_NDA": os.path.join(WORKING, "w7_2_SAFE_for_claude_no_NDA.zip"),
        "3_NEXT_ROUND_PATH_TO_W7_COMMON_SETS_DATASET": os.path.join(WORKING, "w7_3_NEXT_ROUND_PATH_TO_W7_COMMON_SETS_DATASET.zip")}
# make them
for key, items in [("1_ALL_outputs", all_items), ("2_SAFE_for_claude_no_NDA", safe_items), ("3_NEXT_ROUND_PATH_TO_W7_COMMON_SETS_DATASET", next_items)]:
    # build and report the size
    mb = make_zip(ZIPS[key], items)
    # print
    print("%-30s %8.1f MB  %5d files  %s" % (key, mb, len(items), ZIPS[key]))

# download links (Kaggle needs paths relative to /kaggle/working)
try:
    # IPython tools
    from IPython.display import FileLink, display
    # links are relative to the working folder
    os.chdir(WORKING)
    # one link per zip
    for key, path in ZIPS.items():
        # print the label and show the link
        print(key)
        # the link
        display(FileLink(os.path.relpath(path, WORKING)))
# not in a notebook
except Exception as e:
    # just print the paths
    print("no download links here:", e)
# the reminder
if SMOKE:
    print("#" * 78)
    print("#  SMOKE=True: this was only a test. Set SMOKE=False before Save & Run All.")
    print("#" * 78)